# Stain-robust LoRA adaptation of a pathology foundation model for breast cancer histopathology
### Patient-level evaluation on BreakHis with external validation on BACH — final paper notebook (v3.1)

## How to run (3 short runs of THIS notebook; each fits Kaggle's 12-h limit)

| Run | Set in cell 1 | What it does | Time (T4) |
|---|---|---|---|
| 1 | `RUN_PART = 'binary'` | benign vs malignant: all models, 3 LoRA seeds, **BACH external test** | ~7 h |
| 2 | `RUN_PART = 'subtype'` | 8 subtypes: all models, 3 LoRA seeds, stain ablation, leakage demo, saliency maps | ~7 h |
| 3 | `RUN_PART = 'report'` | merges runs 1+2 and builds **every paper table (CSV + LaTeX) and figure (PNG + PDF)** | ~10 min |

**Before every run:** Settings → Accelerator → **GPU T4** · Internet **ON** ·
**Add Input** → BreakHis (`ambarish/breakhis`) and your BACH dataset (`erenyeager00111/iciar2018-bach-images`).
**Before run 3:** also **Add Input → Your Work → this notebook**, and pick the output of run 1 **and** run 2.
(Optional: set `QUICK_TEST = True` first for a ~40-min smoke test.)

## Datasets
- **BreakHis** — 7,909 H&E images, 82 patients, 4 magnifications, 8 subtypes. Spanhol et al., *IEEE Trans.
  Biomed. Eng.* 63(7):1455–1462, 2016, doi:10.1109/TBME.2015.2496264.
  Original: https://web.inf.ufpr.br/vri/databases/breast-cancer-histopathological-database-breakhis/ ·
  Kaggle mirror: https://www.kaggle.com/datasets/ambarish/breakhis
- **BACH (ICIAR 2018)** — 400 microscopy images (Normal, Benign, InSitu, Invasive; 100 each), a different
  centre and scanner. Aresta et al., *Medical Image Analysis* 56:122–139, 2019,
  doi:10.1016/j.media.2019.05.010. Official: https://zenodo.org/records/3632035 ·
  Kaggle copy used here: https://www.kaggle.com/datasets/erenyeager00111/iciar2018-bach-images.
  Used **only for testing**: Normal + Benign → benign, InSitu + Invasive → malignant.

## Method (fixed before results; v3.1 changes only lr 2e-4 → 1e-4, chosen because inner-validation curves peaked at epoch 1–2)
**Ours** = Phikon-v2 ViT-L (Filiot et al., 2024) + **LoRA** (Hu et al., ICLR 2022; rank 8 on Q/K/V of all 24
blocks, ~1.5 M trainable params = 0.5 %) + on-the-fly **H&E stain augmentation** in HED space (Tellez et al.,
*Med. Image Anal.* 58:101544, 2019) + geometric augmentation + class-balanced cross-entropy + 5-view
test-time averaging + temperature scaling fit on the inner validation split.

**Protocol:** exact-duplicate audit → patient-disjoint stratified 5-fold CV (nested patient-level inner
validation for early stopping and calibration) → 3 seeds → paired **patient-cluster bootstrap** 95 % CIs →
external BACH test → automatic **claims checklist** (write only SUPPORTED claims).

**Comparators:** frozen linear probes (DINOv2-B, Phikon-v2), LogReg / RBF-SVM / k-NN / MLP on concatenated
frozen features. Ablations: no stain augmentation. Results already final from the previous run (v3,
not re-run here): fine-tuned ResNet-50 and LoRA-on-DINOv2 ablation — copy them from that run's tables.

## 1. Configuration

In [1]:
import os, sys, re, json, time, math, random, hashlib, subprocess, importlib, warnings, shutil, copy, pickle
from pathlib import Path
from collections import defaultdict

# >>>>>>>>>>>>>>>>>>>>  SET THIS FOR EACH RUN: 'binary'  ->  'subtype'  ->  'report'   (or 'all')  <<<<<<<<<<<<<<<<<<<<
RUN_PART = os.environ.get('CMFE_PART', 'binary')
# >>>>>>>>>>>>>>>>>>>>  True = ~40-min smoke test on a subset (set back to False for the paper run)  <<<<<<<<<<<<<<<<<
QUICK_TEST = False or os.environ.get('CMFE_QUICK', '0') == '1'
DRY_RUN = os.environ.get('CMFE_DRY_RUN', '0') == '1'      # internal code test only — leave as is

BREAKHIS_SLUG = 'ambarish/breakhis'
BACH_SLUG = 'erenyeager00111/iciar2018-bach-images'


def _ensure(pkgs):
    for pip_name, mod in pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

_ensure([('timm', 'timm'), ('transformers', 'transformers'), ('scikit-image', 'skimage')]
        + ([] if DRY_RUN else [('kagglehub', 'kagglehub')]))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import timm
from skimage.color import rgb2hed, hed2rgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score, confusion_matrix,
                             precision_recall_fscore_support)
warnings.filterwarnings('ignore')

WORK = Path(os.environ.get('CMFE_WORK', '/kaggle/working'))
INPUT = Path(os.environ.get('CMFE_INPUT', '/kaggle/input'))
FEAT_DIR, RES_DIR, FIG_DIR, TAB_DIR = WORK / 'features', WORK / 'results', WORK / 'figures', WORK / 'tables'
for d in (FEAT_DIR, RES_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

PARTS = {
    'binary':  dict(tasks=['binary'],            run_external=True,  run_ablation=False, run_leakage_demo=False),
    'subtype': dict(tasks=['subtype'],           run_external=False, run_ablation=True,  run_leakage_demo=True),
    'all':     dict(tasks=['binary', 'subtype'], run_external=True,  run_ablation=True,  run_leakage_demo=True),
    'report':  dict(tasks=[],                    run_external=False, run_ablation=False, run_leakage_demo=False),
}
assert RUN_PART in PARTS, f'RUN_PART must be one of {list(PARTS)}'
REPORT_ONLY = RUN_PART == 'report'

CFG = dict(
    backbones=['dinov2_vitb14', 'phikon_v2'],      # frozen comparators
    n_folds=5, batch_extract=32, num_workers=4,
    # proposed model
    lora_backbone='phikon_v2', lora_r=8, lora_alpha=16, lora_dropout=0.1,
    lora_epochs=6, lora_patience=3, lora_bs=16, lora_lr=1e-4, head_lr=1e-3, lora_wd=1e-4,
    lora_seeds=[0, 1, 2], abl_seeds=[0], ext_seeds=[0, 1, 2], label_smooth=0.1,
    # frozen MLP head
    hid=256, emb=128, epochs=80, patience=15, lr=1e-3, wd=1e-4, batch=256, p_in_drop=0.2,
    # other comparators
    logreg_C=[0.01, 0.1, 1.0], svm_C=10.0, svm_pca=256, knn_k=10,
    n_boot=2000, n_saliency=8,
    **PARTS[RUN_PART],
)
if QUICK_TEST and not DRY_RUN:
    CFG.update(lora_epochs=1, lora_seeds=[0], ext_seeds=[0], epochs=20, patience=5, n_boot=200)
if DRY_RUN:
    CFG.update(n_folds=2, lora_epochs=2, lora_patience=2, lora_bs=8, lora_seeds=[0, 1], ext_seeds=[0],
               epochs=30, patience=10, n_boot=50, batch_extract=8, num_workers=0, svm_pca=16, batch=64, logreg_C=[0.1])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)

set_seed(42)
print(f'RUN_PART={RUN_PART} | QUICK_TEST={QUICK_TEST} | DRY_RUN={DRY_RUN} | device={DEVICE} | torch {torch.__version__}')
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
elif not (DRY_RUN or REPORT_ONLY):
    print('WARNING: no GPU — Settings -> Accelerator -> GPU')
json.dump(CFG, open(RES_DIR / f'config_{RUN_PART}.json', 'w'), indent=2)

RUN_PART=binary | QUICK_TEST=False | DRY_RUN=False | device=cuda | torch 2.11.0+cu128
GPU: Tesla T4


## 2. Dataset download / location

In [2]:
def find_attached(keyword):
    """Folders under /kaggle/input whose name contains keyword (handles both Kaggle input layouts)."""
    if not INPUT.exists():
        return []
    return [d for pat in ('*', '*/*', '*/*/*', '*/*/*/*') for d in INPUT.glob(pat) if d.is_dir() and keyword in d.name.lower()]

if not DRY_RUN:
    import kagglehub
    path = kagglehub.dataset_download(BREAKHIS_SLUG)
    print('BreakHis path:', path)

BreakHis path: /kaggle/input/datasets/ambarish/breakhis


## 3. BreakHis metadata (file names encode class, subtype, patient and magnification)

In [3]:
SUBTYPES = ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
SUB_FULL = {'A': 'Adenosis', 'F': 'Fibroadenoma', 'PT': 'Phyllodes tumour', 'TA': 'Tubular adenoma',
            'DC': 'Ductal carcinoma', 'LC': 'Lobular carcinoma', 'MC': 'Mucinous carcinoma', 'PC': 'Papillary carcinoma'}
MAGS = [40, 100, 200, 400]
RX = re.compile(r'SOB_(?P<cls>[BM])_(?P<sub>[A-Z]+)-(?P<year>\d+)-(?P<pid>[0-9A-Za-z]+)-(?P<mag>\d+)-(?P<seq>\d+)\.png$', re.I)


def _synth_img(rng, si, mag, h=460, w=700):
    base = np.array([200 - 8 * si, 120 + 5 * si, 190 - 10 * si], dtype=np.float32)
    img = base + rng.normal(0, 25, (h, w, 3))
    yy, xx = np.mgrid[0:h, 0:w]
    img[..., 1] += 30 * np.sin(xx / (5 + si) + mag / 100.0)
    return Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))


def make_synthetic(root, n_pat_per_sub=2, n_img=3):
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    rng, pid = np.random.default_rng(0), 1000
    for si, sub in enumerate(SUBTYPES):
        cls = 'B' if si < 4 else 'M'
        for _ in range(n_pat_per_sub):
            pid += 1
            for mag in MAGS:
                d = root / 'histology_slides' / 'breast' / sub / f'SOB_{cls}_{sub}_14-{pid}' / f'{mag}X'
                d.mkdir(parents=True, exist_ok=True)
                for k in range(1, n_img + 1):
                    _synth_img(rng, si, mag).save(d / f'SOB_{cls}_{sub}-14-{pid}-{mag}-{k:03d}.png')
    return root


DATA_ROOT = make_synthetic(WORK / 'synthetic_breakhis') if DRY_RUN else Path(path)
rows, n_bad, seen = [], 0, set()
for p in sorted(DATA_ROOT.rglob('*')):
    if p.suffix.lower() != '.png':
        continue
    m = RX.search(p.name)
    if not m:
        n_bad += 1; continue
    if p.name in seen:
        continue
    seen.add(p.name)
    g = m.groupdict()
    rows.append(dict(path=str(p), file=p.name, subtype=g['sub'].upper(), binary=int(g['cls'].upper() == 'M'),
                     patient=f"{g['year']}-{g['pid']}", mag=int(g['mag'])))
df = pd.DataFrame(rows)
assert len(df), f'No BreakHis PNGs found under {DATA_ROOT}'
df = df[df.subtype.isin(SUBTYPES) & df.mag.isin(MAGS)].reset_index(drop=True)
RAW_STATS = dict(images=len(df), patients=df.patient.nunique())
print(f'images: {len(df)} | patients: {df.patient.nunique()} | unparsed: {n_bad}')
if QUICK_TEST:
    df = df.groupby(['patient', 'mag']).head(4).reset_index(drop=True)
    print('QUICK_TEST subset:', len(df))
print(pd.crosstab(df.subtype, df.mag, margins=True))

images: 7909 | patients: 81 | unparsed: 0
mag        40   100   200   400   All
subtype                              
A         114   113   111   106   444
DC        864   903   896   788  3451
F         253   260   264   237  1014
LC        156   170   163   137   626
MC        205   222   196   169   792
PC        145   142   135   138   560
PT        109   121   108   115   453
TA        149   150   140   130   569
All      1995  2081  2013  1820  7909


## 4. Integrity audit (exact pixel duplicates, before any split)

In [4]:
def pixel_hash(p):
    with Image.open(p) as im:
        return hashlib.md5(np.asarray(im.convert('RGB')).tobytes()).hexdigest()

t0 = time.time()
df['hash'] = [pixel_hash(p) for p in df.path]
drop, audit = set(), []
for h, g in df.groupby('hash'):
    if len(g) < 2:
        continue
    kind = 'cross_label' if g.subtype.nunique() > 1 else ('cross_patient' if g.patient.nunique() > 1 else 'within_patient')
    audit.append(dict(hash=h, kind=kind, n=len(g), patients='|'.join(sorted(set(g.patient))),
                      subtypes='|'.join(sorted(set(g.subtype))), files='|'.join(g.file)))
    drop |= set(g.index) if kind == 'cross_label' else set(g.index[1:])
audit = pd.DataFrame(audit, columns=['hash', 'kind', 'n', 'patients', 'subtypes', 'files'])
audit.to_csv(RES_DIR / 'integrity_audit.csv', index=False)
print(f'hashed in {time.time()-t0:.0f}s | duplicate groups: {len(audit)}', dict(audit.kind.value_counts()) if len(audit) else '')
if len(audit):
    print(audit.groupby(['kind', 'patients', 'subtypes']).size().rename('groups').to_string())
N_DROPPED = len(drop)
df = df.drop(index=sorted(drop)).reset_index(drop=True)
print(f'after cleanup: {len(df)} images ({N_DROPPED} removed)')
df['y_binary'] = df.binary.values
df['y_subtype'] = df.subtype.map({s: i for i, s in enumerate(SUBTYPES)}).values
df['mag_id'] = df.mag.map({m: i for i, m in enumerate(MAGS)}).values
N = len(df)
TASK_C = {'binary': 2, 'subtype': len(SUBTYPES)}
TASK_NAMES = {'binary': ['Benign', 'Malignant'], 'subtype': SUBTYPES}

hashed in 809s | duplicate groups: 125 {'cross_label': np.int64(123), 'within_patient': np.int64(2)}
kind            patients  subtypes
cross_label     14-13412  DC|LC       123
within_patient  14-12204  LC            2
after cleanup: 7661 images (248 removed)


## 5. Patient-disjoint folds (seed 42 — identical to all earlier runs)

In [5]:
pat_label = df.groupby('patient').y_subtype.agg(lambda s: s.mode()[0])
df['strat'] = df.patient.map(pat_label)
df['fold'] = -1
for f, (_, te) in enumerate(StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=42).split(df, df.strat, groups=df.patient)):
    df.loc[te, 'fold'] = f
assert (df.groupby('patient').fold.nunique() == 1).all(), 'patient leakage across folds'
print(pd.crosstab(df.fold, df.subtype))
print('patients per fold:', df.groupby('fold').patient.nunique().to_dict())
df.drop(columns=['hash']).to_csv(RES_DIR / 'metadata_folds.csv', index=False)
PATIENT, PID_INT, MAG, PATHS = df.patient.values, pd.factorize(df.patient.values)[0], df.mag_id.values, df.path.values


def inner_split(tr_idx, seed, groups=None):
    g = PATIENT[tr_idx] if groups is None else groups[tr_idx]
    for k in (5, 4, 3, 2):
        try:
            a, b = next(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(tr_idx, df.strat.values[tr_idx], groups=g))
            return tr_idx[a], tr_idx[b]
        except ValueError:
            continue
    raise RuntimeError('inner split failed')

subtype    A    DC    F   LC   MC   PC   PT   TA
fold                                            
0        121  1129    0  201    0   74    0  124
1          0   300  399    0  179  152  218  129
2        193   506  247   74   90    0  235  197
3          0   577  162  226  459  269    0    0
4        130   816  206    0   64   65    0  119
patients per fold: {0: 17, 1: 15, 2: 15, 3: 16, 4: 17}


## 6. Backbones, stain augmentation, frozen features (comparators)

In [6]:
IMNET = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))


def load_backbone(name):
    """Returns (forward_fn, mean, std, model); forward_fn: Bx3x224x224 normalised -> BxD (CLS)."""
    if name == 'dinov2_vitb14':
        m = timm.create_model('vit_small_patch14_dinov2' if DRY_RUN else 'vit_base_patch14_dinov2.lvd142m',
                              pretrained=not DRY_RUN, num_classes=0, img_size=224)
        return (lambda x: m(x)), IMNET[0], IMNET[1], m
    if name == 'phikon_v2':
        from transformers import AutoModel, AutoImageProcessor, Dinov2Config, Dinov2Model
        if DRY_RUN:
            m = Dinov2Model(Dinov2Config(hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                                         intermediate_size=128, patch_size=16, image_size=224))
            mean, std = IMNET
        else:
            try:
                m = AutoModel.from_pretrained('owkin/phikon-v2', attn_implementation='sdpa')
            except Exception:
                m = AutoModel.from_pretrained('owkin/phikon-v2')
            proc = AutoImageProcessor.from_pretrained('owkin/phikon-v2')
            mean, std = tuple(proc.image_mean), tuple(proc.image_std)
        return (lambda x: m(pixel_values=x).last_hidden_state[:, 0]), mean, std, m
    raise ValueError(name)


def hed_jitter(img, rng, sigma=0.05):
    """HED stain augmentation (Tellez et al. 2019), applied as a delta so zero jitter is the identity."""
    a = np.asarray(img).astype(np.float32) / 255.0
    hed = rgb2hed(a)
    hed_j = hed * rng.uniform(1 - sigma, 1 + sigma, 3) + rng.uniform(-sigma, sigma, 3) * np.abs(hed).max(axis=(0, 1), keepdims=True)
    return Image.fromarray((np.clip(a + hed2rgb(hed_j) - hed2rgb(hed), 0, 1) * 255).astype(np.uint8))


def five_views(img):
    g = TF.to_tensor(img.resize((224, 224), Image.BICUBIC))
    big = TF.to_tensor(img.resize((448, 448), Image.BICUBIC))
    return torch.stack([g] + [big[:, r:r + 224, c:c + 224] for r in (0, 224) for c in (0, 224)])


class ViewDS(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return five_views(Image.open(self.paths[i]).convert('RGB'))


class TrainDS(Dataset):
    def __init__(self, paths, y, stain=True):
        self.paths, self.y, self.stain = list(paths), np.asarray(y), stain
        self.rrc = T.RandomResizedCrop(224, scale=(0.25, 1.0), ratio=(0.75, 1.33))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        rng = np.random.default_rng()
        img = self.rrc(Image.open(self.paths[i]).convert('RGB')).rotate(90 * int(rng.integers(0, 4)))
        if rng.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if self.stain:
            img = TF.adjust_contrast(TF.adjust_brightness(hed_jitter(img, rng), float(rng.uniform(0.9, 1.1))), float(rng.uniform(0.9, 1.1)))
        return TF.to_tensor(img), int(self.y[i])


@torch.no_grad()
def extract_frozen(name, paths, tag):
    cache = FEAT_DIR / f'{name}_{tag}.npz'
    if cache.exists():
        d = np.load(cache, allow_pickle=True)
        if list(d['paths']) == list(paths):
            return d['X']
    fn, mean, std, model = load_backbone(name)
    model.eval().to(DEVICE)
    mean, std = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1), torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)
    feats, t0 = [], time.time()
    for bi, x in enumerate(DataLoader(ViewDS(paths), batch_size=CFG['batch_extract'], num_workers=CFG['num_workers'], pin_memory=AMP)):
        B = x.shape[0]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            f = fn((x.view(B * 5, 3, 224, 224).to(DEVICE) - mean) / std).float().view(B, 5, -1)
        feats.append(torch.cat([f[:, 0], f[:, 1:].mean(1)], 1).cpu().numpy())
        if bi % 60 == 0:
            print(f'  [{name}/{tag}] batch {bi+1}  {time.time()-t0:.0f}s')
    X = np.concatenate(feats).astype(np.float32)
    np.savez(cache, X=X, paths=np.array(list(paths)))
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    print(f'[{name}/{tag}] {X.shape} in {time.time()-t0:.0f}s')
    return X


TAG = 'dry' if DRY_RUN else ('quick' if QUICK_TEST else 'full')
FEATS, BB = [], []
if not REPORT_ONLY:
    for name in CFG['backbones']:
        try:
            FEATS.append(extract_frozen(name, PATHS, TAG)); BB.append(name)
        except Exception as e:
            print(f'!! backbone {name} failed and is skipped: {type(e).__name__}: {e}')
    assert BB, 'no backbone could be loaded (is Internet ON?)'
    print('frozen backbones:', BB, '| dims:', [x.shape[1] for x in FEATS])

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

  [dinov2_vitb14/full] batch 1  6s
  [dinov2_vitb14/full] batch 61  34s
  [dinov2_vitb14/full] batch 121  63s
  [dinov2_vitb14/full] batch 181  93s
[dinov2_vitb14/full] (7661, 1536) in 123s


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

  [phikon_v2/full] batch 1  3s
  [phikon_v2/full] batch 61  81s
  [phikon_v2/full] batch 121  161s
  [phikon_v2/full] batch 181  240s
[phikon_v2/full] (7661, 2048) in 318s
frozen backbones: ['dinov2_vitb14', 'phikon_v2'] | dims: [1536, 2048]


## 7. Proposed model: LoRA-adapted Phikon-v2

In [7]:
class LoRALinear(nn.Module):
    """y = W x + (alpha/r)·B(A(dropout(x))); W frozen; B = 0 at init, so the model starts identical to the base."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.A, self.B = nn.Linear(base.in_features, r, bias=False), nn.Linear(r, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.B(self.A(self.drop(x))) * self.scale


def add_lora(model, r, alpha, dropout, targets=('query', 'key', 'value', 'q_proj', 'k_proj', 'v_proj', 'qkv')):
    """Q/K/V projections: HF old names (query/key/value), HF new names (q/k/v_proj) and timm (qkv)."""
    n = 0
    for module in list(model.modules()):
        for cname, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(module, cname, LoRALinear(child, r, alpha, dropout)); n += 1
    return n


class PeftNet(nn.Module):
    def __init__(self, backbone, n_cls):
        super().__init__()
        fn, mean, std, m = load_backbone(backbone)
        for p in m.parameters():
            p.requires_grad = False
        self.n_lora = add_lora(m, CFG['lora_r'], CFG['lora_alpha'], CFG['lora_dropout'])
        assert self.n_lora > 0, f'LoRA found no attention projections in {backbone}'
        self.m, self.fn, self.backbone = m, fn, backbone
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))
        with torch.no_grad():
            D = fn(torch.zeros(1, 3, 224, 224)).shape[-1]
        self.head = nn.Sequential(nn.LayerNorm(D), nn.Dropout(0.1), nn.Linear(D, n_cls))

    def forward(self, x):
        f = self.fn((x - self.mean) / self.std)
        return self.head(f), f


@torch.no_grad()
def peft_predict(model, paths):
    model.eval()
    L, Fz = [], []
    for x in DataLoader(ViewDS(paths), batch_size=max(1, CFG['lora_bs'] // 2), num_workers=CFG['num_workers'], pin_memory=AMP):
        B = x.shape[0]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            lg, f = model(x.view(B * 5, 3, 224, 224).to(DEVICE))
        L.append(lg.float().view(B, 5, -1).mean(1).cpu()); Fz.append(F.normalize(f.float().view(B, 5, -1).mean(1), dim=1).cpu())
    return torch.cat(L).numpy(), torch.cat(Fz).numpy()


def train_peft(tr, va, y, C, seed, stain=True, epochs=None, paths=None):
    paths = PATHS if paths is None else paths
    set_seed(seed)
    model = PeftNet(CFG['lora_backbone'], C).to(DEVICE)
    lora_p = [p for n, p in model.named_parameters() if p.requires_grad and ('.A.' in n or '.B.' in n)]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not ('.A.' in n or '.B.' in n)]
    n_tr, n_all = sum(p.numel() for p in lora_p + head_p), sum(p.numel() for p in model.m.parameters())
    opt = torch.optim.AdamW([{'params': lora_p, 'lr': CFG['lora_lr']}, {'params': head_p, 'lr': CFG['head_lr']}], weight_decay=CFG['lora_wd'])
    dl = DataLoader(TrainDS(paths[tr], y[tr], stain=stain), batch_size=CFG['lora_bs'], shuffle=True, num_workers=CFG['num_workers'],
                    pin_memory=AMP, drop_last=len(tr) > CFG['lora_bs'], persistent_workers=CFG['num_workers'] > 0)
    E = epochs or CFG['lora_epochs']
    total = max(2, E * len(dl)); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    cnt = np.bincount(y[tr], minlength=C).astype(np.float32)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(np.where(cnt > 0, cnt.sum() / (C * np.maximum(cnt, 1)), 0.0), device=DEVICE),
                             label_smoothing=CFG['label_smooth'])
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    keep_va = va[np.isin(y[va], np.unique(y[tr]))]
    best, best_state, bad, t_start, ep_times = (-1, 1e9), None, 0, time.time(), []
    print(f'    LoRA modules {model.n_lora} | trainable {n_tr/1e6:.2f}M of {n_all/1e6:.1f}M ({100*n_tr/n_all:.2f}%)')
    for ep in range(E):
        model.train(); t0 = time.time()
        for x, yb in dl:
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                loss = ce(model(x.to(DEVICE, non_blocking=True))[0].float(), yb.to(DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        ep_times.append(time.time() - t0)
        if len(keep_va) == 0:
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
            continue
        lv, _ = peft_predict(model, paths[keep_va])
        score = (balanced_accuracy_score(y[keep_va], lv.argmax(1)), F.cross_entropy(torch.tensor(lv), torch.tensor(y[keep_va])).item())
        print(f'    epoch {ep+1}/{E}: val bal_acc {score[0]:.4f}  nll {score[1]:.4f}  ({time.time()-t0:.0f}s)')
        if score[0] > best[0] + 1e-6 or (abs(score[0] - best[0]) <= 1e-6 and score[1] < best[1]):
            best, best_state, bad = score, {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}, 0
        else:
            bad += 1
            if bad >= CFG['lora_patience']:
                break
    model.load_state_dict(best_state, strict=False)
    model.train_info = dict(trainable_M=n_tr / 1e6, backbone_M=n_all / 1e6, epochs_run=ep + 1,
                            sec_per_epoch=float(np.mean(ep_times)), train_min=(time.time() - t_start) / 60)
    return model


def free(model):
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None


def saliency_maps(model, paths, labels, names, out_name):
    """Gradient x activation on the patch tokens entering the last transformer block (HF backbones)."""
    try:
        model.eval()
        imgs, maps, preds = [], [], []
        for p in paths:
            img = Image.open(p).convert('RGB').resize((224, 224), Image.BICUBIC)
            x = TF.to_tensor(img)[None].to(DEVICE)
            out = model.m(pixel_values=(x - model.mean) / model.std, output_hidden_states=True)
            hs = out.hidden_states[-2]
            hs.retain_grad()
            logits = model.head(out.last_hidden_state[:, 0])
            c = int(logits.argmax(1)); preds.append(c)
            model.zero_grad(set_to_none=True); logits[0, c].backward()
            rel = F.relu((hs.grad * hs).sum(-1))[0]
            npatch = (224 // 16) ** 2
            r = rel[-npatch:].view(1, 1, 14, 14)
            r = F.interpolate(r, size=224, mode='bilinear', align_corners=False)[0, 0].detach().cpu().numpy()
            maps.append((r - r.min()) / (r.max() - r.min() + 1e-8)); imgs.append(np.asarray(img))
        n = len(imgs)
        fig, axes = plt.subplots(2, n, figsize=(max(2.2 * n, 6.5), 4.8))
        axes = np.array(axes).reshape(2, n)
        for i in range(n):
            axes[0, i].imshow(imgs[i]); axes[0, i].set_title(f'true {names[labels[i]]}', fontsize=8)
            axes[1, i].imshow(imgs[i]); axes[1, i].imshow(maps[i], cmap='jet', alpha=0.45)
            axes[1, i].set_title(f'pred {names[preds[i]]}', fontsize=8)
            for a in axes[:, i]:
                a.axis('off')
        fig.suptitle('Saliency (gradient × activation, last block) — proposed model', fontsize=10)
        fig.tight_layout(); fig.savefig(FIG_DIR / f'{out_name}.png', dpi=300); fig.savefig(FIG_DIR / f'{out_name}.pdf'); plt.close(fig)
        print('saved', out_name)
    except Exception as e:
        print('saliency skipped:', type(e).__name__, e)

## 8. Comparators and metrics

In [8]:
class MLPHead(nn.Module):
    def __init__(self, D, C):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(CFG['p_in_drop']), nn.Linear(D, CFG['hid']), nn.LayerNorm(CFG['hid']), nn.GELU(), nn.Linear(CFG['hid'], CFG['emb']))
        self.out = nn.Linear(CFG['emb'], C)

    def forward(self, x):
        z = self.net(x)
        return self.out(F.gelu(z)), F.normalize(z, dim=1)


def train_mlp(Xtr, ytr, Xva, yva, C, seed):
    set_seed(seed)
    model = MLPHead(Xtr.shape[1], C).to(DEVICE)
    Xt, yt, Xv = torch.tensor(Xtr, device=DEVICE), torch.tensor(ytr, device=DEVICE), torch.tensor(Xva, device=DEVICE)
    cnt = np.bincount(ytr, minlength=C).astype(np.float32)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(np.where(cnt > 0, cnt.sum() / (C * np.maximum(cnt, 1)), 0.0), device=DEVICE), label_smoothing=CFG['label_smooth'])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    n, bs, keep = len(ytr), min(CFG['batch'], len(ytr)), np.isin(yva, np.unique(ytr))
    best, best_state, bad = -1, None, 0
    for ep in range(CFG['epochs']):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for s in range(max(1, n // bs)):
            b = perm[s * bs:(s + 1) * bs]
            loss = ce(model(Xt[b])[0], yt[b]); opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pv = model(Xv)[0].argmax(1).cpu().numpy()
        sc = balanced_accuracy_score(yva[keep], pv[keep]) if keep.any() else 0
        if sc > best + 1e-6:
            best, best_state, bad = sc, copy.deepcopy(model.state_dict()), 0
        elif ep >= 10:
            bad += 1
            if bad >= CFG['patience']:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def mlp_out(model, X):
    with torch.no_grad():
        lg, z = model(torch.tensor(X, device=DEVICE))
    return lg.cpu().numpy(), z.cpu().numpy()


def full_proba(clf, X, C):
    P = np.zeros((len(X), C)); P[:, clf.classes_] = clf.predict_proba(X)
    return P


def logreg_fit(Xtr, ytr, Xva, yva, C):
    best = None
    for c in CFG['logreg_C']:
        clf = LogisticRegression(C=c, max_iter=3000, class_weight='balanced').fit(Xtr, ytr)
        s = balanced_accuracy_score(yva, full_proba(clf, Xva, C).argmax(1))
        if best is None or s > best[0]:
            best = (s, clf)
    return best[1]


def softmax_np(l):
    e = np.exp(l - l.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits_va, y_va):
    Ts = np.exp(np.linspace(-2.5, 2.5, 101))
    nll = [-np.log(softmax_np(logits_va / t)[np.arange(len(y_va)), y_va] + 1e-12).mean() for t in Ts]
    return Ts[int(np.argmin(nll))]


def ece_score(P, y, n_bins=15):
    conf, pred, e = P.max(1), P.argmax(1), 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e


def macro_auc(y, P):
    if P.shape[1] == 2:
        return roc_auc_score(y, P[:, 1]) if len(np.unique(y)) == 2 else np.nan
    a = [roc_auc_score(y == c, P[:, c]) for c in range(P.shape[1]) if 0 < (y == c).sum() < len(y)]
    return float(np.mean(a)) if a else np.nan


def patient_agg(P, y, pids):
    C = P.shape[1]
    g = pd.DataFrame(P).assign(pid=pids, y=y).groupby('pid')
    return g[list(range(C))].mean().values, g.y.agg(lambda s: s.mode()[0]).values, float(pd.Series(P.argmax(1) == y).groupby(pids).mean().mean())


def all_metrics(P, y, pids, P_raw=None):
    pred = P.argmax(1)
    Pp, yp, prr = patient_agg(P, y, pids)
    out = dict(acc=accuracy_score(y, pred), bal_acc=balanced_accuracy_score(y, pred), macro_f1=f1_score(y, pred, average='macro'),
               auc=macro_auc(y, P), ece=ece_score(P, y), pat_acc=accuracy_score(yp, Pp.argmax(1)),
               pat_bal_acc=balanced_accuracy_score(yp, Pp.argmax(1)), prr=prr)
    if P.shape[1] == 2:
        out['sensitivity'] = float((pred[y == 1] == 1).mean()) if (y == 1).any() else np.nan
        out['specificity'] = float((pred[y == 0] == 0).mean()) if (y == 0).any() else np.nan
    if P_raw is not None:
        out['ece_raw'] = ece_score(P_raw, y)
    return out


ROWS = []
OOF = defaultdict(dict)            # OOF[task][method] -> N x C calibrated probs (seed 0)
OOF_RAW = defaultdict(dict)
OOF_SEEDS = defaultdict(lambda: defaultdict(dict))   # OOF_SEEDS[task][method][seed]
AP_OOF = defaultdict(dict)
EXT_P = defaultdict(dict)          # EXT_P[method][seed] -> probs on BACH
EXT_META = None
TRAIN_INFO = []


def record(exp, task, method, fold, seed, P_te, y_te, pid_te, logit_va=None, y_va=None, logit_te=None, extra=None):
    P_raw = P_te
    if logit_va is not None and logit_te is not None:
        P_te = softmax_np(logit_te / fit_temperature(logit_va, y_va))
    ROWS.append(dict(experiment=exp, task=task, method=method, fold=fold, seed=seed, **all_metrics(P_te, y_te, pid_te, P_raw), **(extra or {})))
    return P_te, P_raw


def retrieval_ap(E, y, pid, k=10):
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    S, ap, pk = E @ E.T, np.full(len(E), np.nan), np.full(len(E), np.nan)
    for i in range(len(E)):
        gi = np.where(pid != pid[i])[0]
        if len(gi) == 0:
            continue
        rel = (y[gi] == y[i])[np.argsort(-S[i, gi])]
        R = int(rel.sum())
        if R:
            pk[i], ap[i] = rel[:k].mean(), (np.cumsum(rel[:R]) / np.arange(1, R + 1) * rel[:R]).sum() / R
    return ap, pk


def add_retrieval(task, method, f, E, te, y):
    ap, pk = retrieval_ap(E, y[te], PATIENT[te])
    ROWS.append(dict(experiment='retrieval', task=task, method=method, fold=f, seed=0, p_at_10=np.nanmean(pk), map_at_r=np.nanmean(ap)))
    AP_OOF[task].setdefault(method, np.full(N, np.nan))[te] = ap


def scale(tr, X):
    sc = StandardScaler().fit(X[tr])
    return sc.transform(X).astype(np.float32), sc

## 9. Main experiment — patient-disjoint 5-fold CV

In [9]:
OURS = 'LoRA Phikon-v2 + stain aug (ours)'
t_main = time.time()
for task in CFG['tasks']:
    y, C = df[f'y_{task}'].values, TASK_C[task]
    for f in range(CFG['n_folds']):
        tr_all, te = np.where(df.fold != f)[0], np.where(df.fold == f)[0]
        tr, va = inner_split(tr_all, seed=f)
        Xs = [scale(tr, X)[0] for X in FEATS]
        Xc = np.concatenate(Xs, 1)
        print(f'\n=== {task} | fold {f} | train {len(tr)} val {len(va)} test {len(te)} ({len(set(PATIENT[te]))} test patients) ===')

        def put(method, s, P, Praw):
            OOF_SEEDS[task][method].setdefault(s, np.zeros((N, C)))[te] = P
            if s == 0:
                OOF[task].setdefault(method, np.zeros((N, C)))[te] = P
                OOF_RAW[task].setdefault(method, np.zeros((N, C)))[te] = Praw

        def rec_sk(method, Pt, Pv):
            put(method, 0, *record('main', task, method, f, 0, Pt, y[te], PATIENT[te], np.log(Pv + 1e-12), y[va], np.log(Pt + 1e-12)))

        for bi, name in enumerate(BB):
            clf = logreg_fit(Xs[bi][tr], y[tr], Xs[bi][va], y[va], C)
            rec_sk(f'LinProbe {name} (frozen)', full_proba(clf, Xs[bi][te], C), full_proba(clf, Xs[bi][va], C))
        clf = logreg_fit(Xc[tr], y[tr], Xc[va], y[va], C)
        rec_sk('LogReg concat (frozen)', full_proba(clf, Xc[te], C), full_proba(clf, Xc[va], C))
        pca = PCA(min(CFG['svm_pca'], len(tr) - 1, Xc.shape[1]), random_state=0).fit(Xc[tr])
        Zp = pca.transform(Xc)
        svm = SVC(C=CFG['svm_C'], gamma='scale', probability=True, class_weight='balanced', random_state=0).fit(Zp[tr], y[tr])
        rec_sk('RBF-SVM concat (frozen)', full_proba(svm, Zp[te], C), full_proba(svm, Zp[va], C))
        knn = KNeighborsClassifier(CFG['knn_k'], metric='cosine', weights='distance').fit(Xc[tr], y[tr])
        rec_sk('k-NN concat (frozen)', full_proba(knn, Xc[te], C), full_proba(knn, Xc[va], C))
        for s in CFG['lora_seeds']:
            mlp = train_mlp(Xc[tr], y[tr], Xc[va], y[va], C, s)
            (lv, _), (lt, zt) = mlp_out(mlp, Xc[va]), mlp_out(mlp, Xc[te])
            put('MLP head concat (frozen)', s, *record('main', task, 'MLP head concat (frozen)', f, s, softmax_np(lt), y[te], PATIENT[te], lv, y[va], lt))
            if s == 0:
                add_retrieval(task, 'MLP head concat (frozen)', f, zt, te, y)
        add_retrieval(task, 'raw concat (frozen)', f, Xc[te], te, y)
        for bi, name in enumerate(BB):
            add_retrieval(task, f'raw {name} (frozen)', f, Xs[bi][te], te, y)

        for s in CFG['lora_seeds']:
            print(f'  -- OURS seed {s}')
            model = train_peft(tr, va, y, C, seed=s)
            lv, _ = peft_predict(model, PATHS[va])
            lt, ft = peft_predict(model, PATHS[te])
            TRAIN_INFO.append(dict(task=task, fold=f, seed=s, method=OURS, **model.train_info))
            if f == 0 and s == CFG['lora_seeds'][0]:
                pick = []
                for c in range(C):
                    idx = te[(y[te] == c) & (lt.argmax(1) == c)]
                    if len(idx):
                        pick.append(idx[0])
                pick = pick[:CFG['n_saliency']]
                if pick:
                    saliency_maps(model, PATHS[pick], y[pick], TASK_NAMES[task], f'F14_saliency_{task}')
            free(model)
            put(OURS, s, *record('main', task, OURS, f, s, softmax_np(lt), y[te], PATIENT[te], lv, y[va], lt))
            if s == CFG['lora_seeds'][0]:
                add_retrieval(task, OURS, f, ft, te, y)
        pd.DataFrame(ROWS).to_csv(RES_DIR / f'all_rows_{RUN_PART}.csv', index=False)
if CFG['tasks']:
    print(f'\nmain experiment done in {(time.time()-t_main)/60:.1f} min')


=== binary | fold 0 | train 4909 val 1103 test 1649 (17 test patients) ===
  -- OURS seed 0


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.9590  nll 0.1868  (170s)
    epoch 2/6: val bal_acc 0.9656  nll 0.1753  (169s)
    epoch 3/6: val bal_acc 0.9666  nll 0.1634  (169s)
    epoch 4/6: val bal_acc 0.9640  nll 0.1795  (170s)
    epoch 5/6: val bal_acc 0.9699  nll 0.1651  (170s)
    epoch 6/6: val bal_acc 0.9662  nll 0.1725  (170s)
saved F14_saliency_binary
  -- OURS seed 1


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.9601  nll 0.1831  (170s)
    epoch 2/6: val bal_acc 0.9349  nll 0.2460  (169s)
    epoch 3/6: val bal_acc 0.9338  nll 0.2464  (170s)
    epoch 4/6: val bal_acc 0.9613  nll 0.1875  (169s)
    epoch 5/6: val bal_acc 0.9621  nll 0.1894  (170s)
    epoch 6/6: val bal_acc 0.9584  nll 0.1894  (170s)
  -- OURS seed 2


...

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.8944  nll 0.2010  (186s)
    epoch 2/6: val bal_acc 0.8921  nll 0.2028  (186s)
    epoch 3/6: val bal_acc 0.8902  nll 0.1915  (186s)
    epoch 4/6: val bal_acc 0.8962  nll 0.2067  (186s)
    epoch 5/6: val bal_acc 0.8942  nll 0.1960  (186s)
    epoch 6/6: val bal_acc 0.8938  nll 0.1987  (186s)
  -- OURS seed 1


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.9401  nll 0.1833  (186s)
    epoch 2/6: val bal_acc 0.9304  nll 0.2219  (186s)
    epoch 3/6: val bal_acc 0.9344  nll 0.2254  (186s)
    epoch 4/6: val bal_acc 0.9318  nll 0.2013  (186s)
  -- OURS seed 2


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.9077  nll 0.1763  (186s)
    epoch 2/6: val bal_acc 0.9010  nll 0.2047  (186s)
    epoch 3/6: val bal_acc 0.9253  nll 0.2010  (186s)
    epoch 4/6: val bal_acc 0.9021  nll 0.1982  (186s)
    epoch 5/6: val bal_acc 0.9052  nll 0.1923  (186s)
    epoch 6/6: val bal_acc 0.9039  nll 0.2056  (186s)

main experiment done in 288.3 min


## 10. Ablation (subtype): no stain augmentation

In [10]:
ABL_NAME = 'Abl: no stain aug'
if CFG['run_ablation']:
    task = 'subtype'
    y, C = df[f'y_{task}'].values, TASK_C[task]
    for f in range(CFG['n_folds']):
        tr_all, te = np.where(df.fold != f)[0], np.where(df.fold == f)[0]
        tr, va = inner_split(tr_all, seed=f)
        for s in CFG['abl_seeds']:
            print(f'\n--- {ABL_NAME} | fold {f} | seed {s}')
            model = train_peft(tr, va, y, C, seed=s, stain=False)
            lv, _ = peft_predict(model, PATHS[va]); lt, _ = peft_predict(model, PATHS[te])
            free(model)
            P, Praw = record('ablation', task, ABL_NAME, f, s, softmax_np(lt), y[te], PATIENT[te], lv, y[va], lt)
            OOF_SEEDS[task][ABL_NAME].setdefault(s, np.zeros((N, C)))[te] = P
            if s == 0:
                OOF[task].setdefault(ABL_NAME, np.zeros((N, C)))[te] = P
    pd.DataFrame(ROWS).to_csv(RES_DIR / f'all_rows_{RUN_PART}.csv', index=False)

## 11. Leakage demonstration (same models, image-level vs patient-level split)

In [11]:
if CFG['run_leakage_demo']:
    task = 'subtype'
    y, C = df[f'y_{task}'].values, TASK_C[task]
    img_folds = np.zeros(N, int)
    for f, (_, te) in enumerate(StratifiedKFold(CFG['n_folds'], shuffle=True, random_state=42).split(df, y)):
        img_folds[te] = f
    for split_name, folds in [('patient-level', df.fold.values), ('image-level (leaky)', img_folds)]:
        for f in range(CFG['n_folds']):
            tr, te = np.where(folds != f)[0], np.where(folds == f)[0]
            Xc = np.concatenate([scale(tr, X)[0] for X in FEATS], 1)
            clf = LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced').fit(Xc[tr], y[tr])
            record('leakage', task, 'LogReg concat (frozen)', f, 0, full_proba(clf, Xc[te], C), y[te], PATIENT[te], extra=dict(split=split_name))
            knn = KNeighborsClassifier(CFG['knn_k'], metric='cosine', weights='distance').fit(Xc[tr], y[tr])
            record('leakage', task, 'k-NN concat (frozen)', f, 0, full_proba(knn, Xc[te], C), y[te], PATIENT[te], extra=dict(split=split_name))
        print(f'leakage demo: {split_name} done')
    pd.DataFrame(ROWS).to_csv(RES_DIR / f'all_rows_{RUN_PART}.csv', index=False)

## 12. External validation — BACH (ICIAR 2018), trained on BreakHis only
Dataset: https://www.kaggle.com/datasets/erenyeager00111/iciar2018-bach-images (official: https://zenodo.org/records/3632035).
Normal + Benign → benign (0); InSitu + Invasive → malignant (1). No BACH image is used for training or model selection.

In [12]:
EXT_LABEL = {'normal': 0, 'benign': 0, 'insitu': 1, 'in situ': 1, 'invasive': 1}


def make_synthetic_bach(root, n=8):
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    rng = np.random.default_rng(1)
    for cls, si in [('Normal', 1), ('Benign', 2), ('InSitu', 5), ('Invasive', 6)]:
        d = root / 'Photos' / cls
        d.mkdir(parents=True, exist_ok=True)
        for k in range(n):
            _synth_img(rng, si, 200, 384, 512).save(d / f'{cls[:2].lower()}{k:03d}.tif')
    return root


def scan_external(root):
    er = []
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() not in ('.tif', '.tiff', '.png', '.jpg', '.jpeg') or 'mask' in str(p).lower():
            continue
        if RX.search(p.name) or 'breakhis' in str(p).lower():
            continue
        par = [a.name for a in p.parents if a.name.lower() in EXT_LABEL]
        if par and p.stat().st_size < 80e6:
            er.append(dict(path=str(p), y=EXT_LABEL[par[0].lower()], cls=par[0]))
    return pd.DataFrame(er, columns=['path', 'y', 'cls']).drop_duplicates('path')


EXT = None
if CFG['run_external']:
    try:
        if DRY_RUN:
            EXT = scan_external(make_synthetic_bach(WORK / 'synthetic_bach'))
        else:
            roots = find_attached('bach')
            print('attached BACH folders:', [str(r) for r in roots])
            EXT = pd.concat([scan_external(r) for r in roots]).drop_duplicates('path') if roots else pd.DataFrame()
            if len(EXT) == 0:
                import kagglehub
                EXT = scan_external(kagglehub.dataset_download(BACH_SLUG))
        if QUICK_TEST and len(EXT):
            EXT = EXT.groupby('cls').head(20).reset_index(drop=True)
        print('external images:', len(EXT), dict(EXT.cls.value_counts()) if len(EXT) else '')
        assert len(EXT) > 0 and EXT.y.nunique() == 2
    except Exception as e:
        print(f'!! BACH unavailable — attach it with Add Input ({BACH_SLUG}): {type(e).__name__}: {e}')
        EXT = None

if EXT is not None:
    EXT_META = EXT.reset_index(drop=True)
    y_ext, ext_pid = EXT_META.y.values, np.arange(len(EXT_META)).astype(str)
    y = df.y_binary.values
    tr, va = inner_split(np.arange(N), seed=0)
    Xs, Es = [], []
    for name, X in zip(BB, FEATS):
        xs, sc = scale(tr, X)
        Xs.append(xs); Es.append(sc.transform(extract_frozen(name, EXT_META.path.values, f'bach_{TAG}')).astype(np.float32))
    Xc, Ec = np.concatenate(Xs, 1), np.concatenate(Es, 1)

    def rec_ext(method, s, Pt, Pv, logits=False):
        lv, lt = (Pv, Pt) if logits else (np.log(Pv + 1e-12), np.log(Pt + 1e-12))
        P, _ = record('external', 'binary', method, -1, s, softmax_np(lt), y_ext, ext_pid, lv, y[va], lt)
        EXT_P[method][s] = P

    for bi, name in enumerate(BB):
        clf = logreg_fit(Xs[bi][tr], y[tr], Xs[bi][va], y[va], 2)
        rec_ext(f'LinProbe {name} (frozen)', 0, full_proba(clf, Es[bi], 2), full_proba(clf, Xs[bi][va], 2))
    clf = logreg_fit(Xc[tr], y[tr], Xc[va], y[va], 2)
    rec_ext('LogReg concat (frozen)', 0, full_proba(clf, Ec, 2), full_proba(clf, Xc[va], 2))
    pca = PCA(min(CFG['svm_pca'], len(tr) - 1, Xc.shape[1]), random_state=0).fit(Xc[tr])
    svm = SVC(C=CFG['svm_C'], gamma='scale', probability=True, class_weight='balanced', random_state=0).fit(pca.transform(Xc[tr]), y[tr])
    rec_ext('RBF-SVM concat (frozen)', 0, full_proba(svm, pca.transform(Ec), 2), full_proba(svm, pca.transform(Xc[va]), 2))
    knn = KNeighborsClassifier(CFG['knn_k'], metric='cosine', weights='distance').fit(Xc[tr], y[tr])
    rec_ext('k-NN concat (frozen)', 0, full_proba(knn, Ec, 2), full_proba(knn, Xc[va], 2))
    for s in CFG['ext_seeds']:
        mlp = train_mlp(Xc[tr], y[tr], Xc[va], y[va], 2, s)
        rec_ext('MLP head concat (frozen)', s, mlp_out(mlp, Ec)[0], mlp_out(mlp, Xc[va])[0], logits=True)
    for name, kw, seeds in [(OURS, {}, CFG['ext_seeds']), (ABL_NAME, dict(stain=False), CFG['ext_seeds'][:1])]:
        for s in seeds:
            print(f'\n--- external: {name} seed {s}')
            model = train_peft(tr, va, y, 2, seed=s, **kw)
            lv, _ = peft_predict(model, PATHS[va]); lt, _ = peft_predict(model, EXT_META.path.values)
            free(model)
            rec_ext(name, s, lt, lv, logits=True)
    print('external validation done')

attached BACH folders: ['/kaggle/input/datasets/erenyeager00111/iciar2018-bach-images']
external images: 400 {'Benign': np.int64(100), 'InSitu': np.int64(100), 'Invasive': np.int64(100), 'Normal': np.int64(100)}
  [dinov2_vitb14/bach_full] batch 1  13s
[dinov2_vitb14/bach_full] (400, 1536) in 41s


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

  [phikon_v2/bach_full] batch 1  6s
[phikon_v2/bach_full] (400, 2048) in 21s

--- external: LoRA Phikon-v2 + stain aug (ours) seed 0


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.7658  nll 0.3784  (225s)
    epoch 2/6: val bal_acc 0.7645  nll 0.3815  (225s)
    epoch 3/6: val bal_acc 0.7669  nll 0.3849  (225s)
    epoch 4/6: val bal_acc 0.7547  nll 0.3549  (225s)
    epoch 5/6: val bal_acc 0.7666  nll 0.3523  (225s)
    epoch 6/6: val bal_acc 0.7543  nll 0.3445  (225s)

--- external: LoRA Phikon-v2 + stain aug (ours) seed 1


...

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.7781  nll 0.3752  (225s)
    epoch 2/6: val bal_acc 0.7406  nll 0.3193  (226s)
    epoch 3/6: val bal_acc 0.7976  nll 0.4267  (225s)
    epoch 4/6: val bal_acc 0.7596  nll 0.4025  (225s)
    epoch 5/6: val bal_acc 0.7595  nll 0.3212  (225s)
    epoch 6/6: val bal_acc 0.7648  nll 0.3446  (225s)

--- external: LoRA Phikon-v2 + stain aug (ours) seed 2


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.7320  nll 0.3161  (225s)
    epoch 2/6: val bal_acc 0.7531  nll 0.3583  (225s)
    epoch 3/6: val bal_acc 0.7723  nll 0.3313  (225s)
    epoch 4/6: val bal_acc 0.7614  nll 0.3312  (225s)
    epoch 5/6: val bal_acc 0.7571  nll 0.3106  (225s)
    epoch 6/6: val bal_acc 0.7507  nll 0.3224  (225s)

--- external: Abl: no stain aug seed 0


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.18M of 304.5M (0.39%)
    epoch 1/6: val bal_acc 0.7892  nll 0.3750  (224s)
    epoch 2/6: val bal_acc 0.7809  nll 0.3714  (225s)
    epoch 3/6: val bal_acc 0.7630  nll 0.3774  (224s)
    epoch 4/6: val bal_acc 0.7747  nll 0.3598  (225s)
external validation done


## 13. Save this run's bundle (run 3 merges the bundles of runs 1 and 2)

In [13]:
def to_plain(d):
    return {k: (to_plain(v) if isinstance(v, (dict, defaultdict)) else v) for k, v in d.items()}

if not REPORT_ONLY:
    bundle = dict(part=RUN_PART, cfg=CFG, rows=ROWS, oof=to_plain(OOF), oof_raw=to_plain(OOF_RAW), oof_seeds=to_plain(OOF_SEEDS),
                  ap_oof=to_plain(AP_OOF), ext_p=to_plain(EXT_P), ext_meta=EXT_META, train_info=TRAIN_INFO, files=list(df.file))
    with open(RES_DIR / f'cmfe_part_{RUN_PART}.pkl', 'wb') as fh:
        pickle.dump(bundle, fh)
    print('saved bundle', RES_DIR / f'cmfe_part_{RUN_PART}.pkl')

saved bundle /kaggle/working/results/cmfe_part_binary.pkl


## 14. Merge all available bundles (this run + attached outputs of earlier runs)

In [14]:
bundle_files = sorted(set(list(RES_DIR.glob('cmfe_part_*.pkl')) + (list(INPUT.rglob('cmfe_part_*.pkl')) if INPUT.exists() else [])))
print('bundles found:', [str(b) for b in bundle_files])
B_ALL = {}
for bf in bundle_files:
    b = pickle.load(open(bf, 'rb'))
    if b['files'] != list(df.file):
        print(f'!! {bf} was made on a different image list (QUICK_TEST mismatch?) — skipped'); continue
    B_ALL[b['part']] = b          # later files with the same part overwrite earlier ones
assert B_ALL, 'no results: run RUN_PART="binary" and "subtype" first (and attach their outputs for "report")'
ROWS, TRAIN_INFO = [], []
OOF, OOF_RAW, AP_OOF = defaultdict(dict), defaultdict(dict), defaultdict(dict)
OOF_SEEDS, EXT_P = defaultdict(lambda: defaultdict(dict)), defaultdict(dict)
for part, b in B_ALL.items():
    ROWS += b['rows']; TRAIN_INFO += b['train_info']
    for task, d in b['oof'].items():
        OOF[task].update(d)
    for task, d in b['oof_raw'].items():
        OOF_RAW[task].update(d)
    for task, d in b['ap_oof'].items():
        AP_OOF[task].update(d)
    for task, d in b['oof_seeds'].items():
        for m, ds in d.items():
            OOF_SEEDS[task][m].update(ds)
    for m, ds in b['ext_p'].items():
        EXT_P[m].update(ds)
    if b['ext_meta'] is not None:
        EXT_META = b['ext_meta']
R = pd.DataFrame(ROWS).drop_duplicates(subset=[c for c in ['experiment', 'task', 'method', 'fold', 'seed', 'split'] if c in pd.DataFrame(ROWS).columns])
R.to_csv(RES_DIR / 'all_rows_merged.csv', index=False)
TASKS_DONE = [t for t in ['binary', 'subtype'] if OURS in OOF.get(t, {})]
print('parts merged:', list(B_ALL), '| tasks with results:', TASKS_DONE, '| BACH:', EXT_META is not None)

bundles found: ['/kaggle/working/results/cmfe_part_binary.pkl']
parts merged: ['binary'] | tasks with results: ['binary'] | BACH: True


## 15. Paper tables (CSV + LaTeX in /kaggle/working/tables)

In [15]:
pd.set_option('display.width', 250)


def save_table(t, name, caption=''):
    t.to_csv(TAB_DIR / f'{name}.csv')
    try:
        t.to_latex(TAB_DIR / f'{name}.tex', caption=caption, label=f'tab:{name}', escape=True)
    except Exception as e:
        print('latex export skipped:', e)
    print(f'\n=== {name}: {caption} ===\n{t.to_string()}')


PCT = ['acc', 'bal_acc', 'macro_f1', 'auc', 'sensitivity', 'specificity', 'pat_acc', 'pat_bal_acc', 'prr']
NICE = dict(acc='Accuracy', bal_acc='Balanced acc.', macro_f1='Macro-F1', auc='AUC', sensitivity='Sensitivity', specificity='Specificity',
            pat_acc='Patient acc.', pat_bal_acc='Patient bal. acc.', prr='PRR', ece='ECE (calibrated)', ece_raw='ECE (raw)')


def summarize(sub):
    ks = [k for k in PCT + ['ece', 'ece_raw'] if k in sub and sub[k].notna().any()]
    agg = sub.groupby('method')[ks].agg(['mean', 'std'])
    out = pd.DataFrame(index=agg.index)
    for k in ks:
        out[NICE[k]] = [(f'{m*100:.2f} ± {0 if np.isnan(s) else s*100:.2f}' if k in PCT else f'{m:.3f} ± {0 if np.isnan(s) else s:.3f}') for m, s in agg[k].values]
    return out.loc[sub.groupby('method').bal_acc.mean().sort_values(ascending=False).index]


# T1 dataset
t1 = pd.crosstab(df.subtype.map(lambda s: f'{SUB_FULL[s]} ({s})'), df.mag, margins=True, margins_name='Total')
t1['Patients'] = df.groupby(df.subtype.map(lambda s: f'{SUB_FULL[s]} ({s})')).patient.nunique().reindex(t1.index).fillna(df.patient.nunique()).astype(int)
save_table(t1, 'T1_dataset', f'BreakHis after integrity cleanup ({N_DROPPED} duplicate images removed); images per magnification and patients per subtype.')
if len(audit):
    save_table(audit.groupby(['kind', 'patients', 'subtypes']).agg(groups=('n', 'size'), images=('n', 'sum')), 'T2_integrity_audit', 'Exact pixel duplicates found in BreakHis.')
main = R[R.experiment == 'main']
for i, task in enumerate(TASKS_DONE):
    save_table(summarize(main[main.task == task]), f'T{3+i}_main_{task}',
               f'{task}: patient-disjoint 5-fold CV, mean ± std over folds (× seeds for stochastic models), %.')
if 'ablation' in set(R.experiment):
    save_table(summarize(pd.concat([main[(main.task == 'subtype') & (main.method == OURS)], R[R.experiment == 'ablation']])), 'T5_ablation',
               'Ablation on the subtype task.')
if 'external' in set(R.experiment):
    save_table(summarize(R[R.experiment == 'external']), 'T6_external_bach', 'External validation on BACH (trained on BreakHis only), %.')
if 'leakage' in set(R.experiment):
    lk = (R[R.experiment == 'leakage'].pivot_table(index='method', columns='split', values=['acc', 'bal_acc'], aggfunc='mean') * 100).round(2)
    save_table(lk, 'T7_leakage', 'Same models evaluated with image-level vs patient-level splits (subtype), %.')
if 'retrieval' in set(R.experiment):
    save_table((R[R.experiment == 'retrieval'].pivot_table(index=['task', 'method'], values=['map_at_r', 'p_at_10'], aggfunc='mean') * 100).round(2),
               'T8_retrieval', 'Cross-patient retrieval on unseen patients (mAP@R, P@10), %.')
# T9 per magnification (seed 0, pooled OOF)
pm = []
for task in TASKS_DONE:
    y = df[f'y_{task}'].values
    for m, P in OOF[task].items():
        for mi, mag in enumerate(MAGS):
            k = MAG == mi
            pm.append(dict(task=task, method=m, mag=f'{mag}X', bal_acc=balanced_accuracy_score(y[k], P[k].argmax(1)) * 100))
if pm:
    pm = pd.DataFrame(pm)
    save_table(pm.pivot_table(index=['task', 'method'], columns='mag', values='bal_acc').round(2), 'T9_per_magnification',
               'Balanced accuracy per magnification (pooled out-of-fold, seed 0), %.')
# T10 per class (ours)
for task in TASKS_DONE:
    y = df[f'y_{task}'].values
    pr, rc, f1, sp = precision_recall_fscore_support(y, OOF[task][OURS].argmax(1), labels=range(TASK_C[task]), zero_division=0)
    names = [SUB_FULL.get(n, n) for n in TASK_NAMES[task]]
    save_table(pd.DataFrame(dict(precision=pr * 100, recall=rc * 100, f1=f1 * 100, images=sp), index=names).round(2),
               f'T10_per_class_{task}', f'Per-class performance of the proposed model ({task}, pooled out-of-fold, seed 0), %.')
# T11 compute
if TRAIN_INFO:
    save_table(pd.DataFrame(TRAIN_INFO).groupby(['task', 'method'])[['trainable_M', 'backbone_M', 'epochs_run', 'sec_per_epoch', 'train_min']].mean().round(2),
               'T11_compute', 'Training cost of the proposed model per fold (Tesla T4).')


=== T1_dataset: BreakHis after integrity cleanup (248 duplicate images removed); images per magnification and patients per subtype. ===
mag                         40   100   200   400  Total  Patients
subtype                                                          
Adenosis (A)               114   113   111   106    444         4
Ductal carcinoma (DC)      832   870   864   762   3328        37
Fibroadenoma (F)           253   260   264   237   1014        10
Lobular carcinoma (LC)     122   137   131   111    501         4
Mucinous carcinoma (MC)    205   222   196   169    792         9
Papillary carcinoma (PC)   145   142   135   138    560         6
Phyllodes tumour (PT)      109   121   108   115    453         3
Tubular adenoma (TA)       149   150   140   130    569         7
Total                     1929  2015  1949  1768   7661        80

=== T2_integrity_audit: Exact pixel duplicates found in BreakHis. ===
                                  groups  images
kind           pa

## 16. Significance tests (paired patient-cluster bootstrap, per seed) and claims checklist

In [16]:
def bal_acc_fast(y, pred):
    return float(np.mean([(pred[y == c] == c).mean() for c in np.unique(y)]))


def cluster_bootstrap(stat_fn, pids, n_boot, seed=0):
    rng, up = np.random.default_rng(seed), np.unique(pids)
    groups = pd.Series(np.arange(len(pids))).groupby(pids).apply(np.array).to_dict()
    obs = stat_fn(np.arange(len(pids)))
    diffs = np.array([stat_fn(np.concatenate([groups[p] for p in rng.choice(up, len(up))])) for _ in range(n_boot)])
    return obs, np.percentile(diffs, 2.5), np.percentile(diffs, 97.5), min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))


def compare(analysis, task, yv, ours_by_seed, comp_by_seed, pids, comp):
    out = []
    for s, Po in ours_by_seed.items():
        Pc = comp_by_seed.get(s, comp_by_seed[min(comp_by_seed)])        # deterministic comparators: seed 0
        a, b = Po.argmax(1), Pc.argmax(1)
        d, lo_, hi_, p = cluster_bootstrap(lambda ii: bal_acc_fast(yv[ii], a[ii]) - bal_acc_fast(yv[ii], b[ii]), pids, CFG['n_boot'], seed=s)
        out.append(dict(seed=s, d=d * 100, lo=lo_ * 100, hi=hi_ * 100, p=p))
    o = pd.DataFrame(out)
    n_sig = int((o.lo > 0).sum()); n_worse = int((o.hi < 0).sum())
    return dict(analysis=analysis, task=task, comparator=comp, delta_pp_mean=o.d.mean(), ci95_lo_seed0=o.lo.iloc[0], ci95_hi_seed0=o.hi.iloc[0],
                p_seed0=o.p.iloc[0], seeds_ours_better=f'{n_sig}/{len(o)}', seeds_comp_better=f'{n_worse}/{len(o)}',
                verdict='OURS better (all seeds)' if n_sig == len(o) else ('comparator better' if n_worse == len(o) else
                         ('OURS better (some seeds)' if n_sig else 'no significant difference')))


sig = []
for task in TASKS_DONE:
    yv = df[f'y_{task}'].values
    for m in OOF_SEEDS[task]:
        if m != OURS:
            sig.append(compare('classification', task, yv, OOF_SEEDS[task][OURS], OOF_SEEDS[task][m], PATIENT, m))
    A = AP_OOF[task].get(OURS)
    for m, Bv in AP_OOF[task].items():
        if A is not None and m != OURS:
            ok = ~np.isnan(A) & ~np.isnan(Bv)
            if ok.sum() >= 10:
                a_, b_ = A[ok], Bv[ok]
                d, lo_, hi_, p = cluster_bootstrap(lambda ii: a_[ii].mean() - b_[ii].mean(), PATIENT[ok], CFG['n_boot'])
                sig.append(dict(analysis='retrieval mAP@R', task=task, comparator=m, delta_pp_mean=d * 100, ci95_lo_seed0=lo_ * 100,
                                ci95_hi_seed0=hi_ * 100, p_seed0=p, verdict='OURS better (all seeds)' if lo_ > 0 else ('comparator better' if hi_ < 0 else 'no significant difference')))
if EXT_META is not None and OURS in EXT_P:
    yx = EXT_META.y.values
    for m in EXT_P:
        if m != OURS:
            sig.append(compare('external BACH', 'binary', yx, EXT_P[OURS], EXT_P[m], np.arange(len(yx)), m))
sig = pd.DataFrame(sig)
if len(sig):
    save_table(sig.set_index(['analysis', 'task', 'comparator']).round(4), 'T12_significance',
               'Proposed model vs each comparator: Δ balanced accuracy (pp), patient-cluster bootstrap 95% CI (seed 0) and number of seeds with a significant win.')


def claim(text, ok, evidence):
    print(f"[{'SUPPORTED    ' if ok else 'NOT SUPPORTED'}] {text}\n      evidence: {evidence}")
    return dict(claim=text, supported=bool(ok), evidence=evidence)


def ev(s):
    return '; '.join(f'{r.comparator}: {r.delta_pp_mean:+.2f}pp, seed0 CI [{r.ci95_lo_seed0:.2f},{r.ci95_hi_seed0:.2f}], wins {getattr(r, "seeds_ours_better", "-")}' for r in s.itertuples())


claims = []
if len(sig):
    for task in TASKS_DONE:
        s = sig[(sig.analysis == 'classification') & (sig.task == task) & ~sig.comparator.str.startswith('Abl')]
        if len(s):
            claims.append(claim(f'[{task}] OURS beats EVERY comparator in all seeds', (s.verdict == 'OURS better (all seeds)').all(), ev(s)))
            lp = s[s.comparator == 'LinProbe phikon_v2 (frozen)']
            if len(lp):
                claims.append(claim(f'[{task}] LoRA adaptation beats the SAME backbone frozen (linear probe), all seeds',
                                    lp.verdict.iloc[0] == 'OURS better (all seeds)', ev(lp)))
            claims.append(claim(f'[{task}] OURS is never significantly worse than any comparator', ~s.verdict.str.startswith('comparator').any(), ev(s)))
    r = sig[(sig.analysis == 'classification') & (sig.comparator == ABL_NAME)]
    if len(r):
        claims.append(claim('[subtype] stain augmentation improves accuracy', r.verdict.iloc[0].startswith('OURS better'), ev(r)))
    s = sig[(sig.analysis == 'external BACH') & ~sig.comparator.str.startswith('Abl')]
    if len(s):
        claims.append(claim('[BACH external] OURS beats EVERY comparator in all seeds', (s.verdict == 'OURS better (all seeds)').all(), ev(s)))
        claims.append(claim('[BACH external] OURS is never significantly worse than any comparator', ~s.verdict.str.startswith('comparator').any(), ev(s)))
    r = sig[(sig.analysis == 'external BACH') & (sig.comparator == ABL_NAME)]
    if len(r):
        claims.append(claim('[BACH external] stain augmentation improves cross-site generalisation', r.verdict.iloc[0].startswith('OURS better'), ev(r)))
if 'leakage' in set(R.experiment):
    lk = R[R.experiment == 'leakage'].groupby('split').bal_acc.mean() * 100
    claims.append(claim('Image-level splitting inflates BreakHis subtype accuracy by >20 pp', lk.get('image-level (leaky)', 0) - lk.get('patient-level', 0) > 20, lk.round(2).to_dict()))
if len(audit):
    claims.append(claim('BreakHis contains cross-label duplicate images', (audit.kind == 'cross_label').any(), audit.groupby(['kind', 'patients', 'subtypes']).size().to_dict()))
pd.DataFrame(claims).to_csv(TAB_DIR / 'claims_checklist.csv', index=False)


=== T12_significance: Proposed model vs each comparator: Δ balanced accuracy (pp), patient-cluster bootstrap 95% CI (seed 0) and number of seeds with a significant win. ===
                                                        delta_pp_mean  ci95_lo_seed0  ci95_hi_seed0  p_seed0 seeds_ours_better seeds_comp_better                    verdict
analysis        task   comparator                                                                                                                                          
classification  binary LinProbe dinov2_vitb14 (frozen)         4.3302         2.2580         8.0864    0.000               2/3               0/3   OURS better (some seeds)
                       LinProbe phikon_v2 (frozen)             0.3756        -0.3388         2.3389    0.125               0/3               0/3  no significant difference
                       LogReg concat (frozen)                  0.4309        -0.8463         2.8758    0.255               0/3            

## 17. Paper figures (PNG 300 dpi + PDF in /kaggle/working/figures)

In [17]:
OKABE = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442', '#000000']
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'font.family': 'DejaVu Sans'})


def save(fig, name):
    fig.tight_layout(); fig.savefig(FIG_DIR / f'{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.close(fig); print('saved', name)


def short(m):
    return m.replace(' (frozen)', '*').replace('LoRA Phikon-v2 + stain aug (ours)', 'Ours (LoRA Phikon-v2)')


def hbar(sub, metric, name, xlabel):
    s = sub.groupby('method')[metric].agg(['mean', 'std']).sort_values('mean')
    fig, ax = plt.subplots(figsize=(7.6, 0.42 * len(s) + 1.3))
    ax.barh([short(i) for i in s.index], s['mean'] * 100, xerr=s['std'].fillna(0) * 100, capsize=2,
            color=[OKABE[3] if i == OURS else (OKABE[5] if i.startswith('Abl') else OKABE[0]) for i in s.index])
    for yi, (mv, sv) in enumerate(zip(s['mean'], s['std'].fillna(0))):
        ax.text(mv * 100 + sv * 100 + 0.4, yi, f'{mv*100:.1f}', va='center', fontsize=8)
    ax.set_xlabel(xlabel); ax.set_xlim(max(0, s['mean'].min() * 100 - 12), min(100, s['mean'].max() * 100 + 8))
    ax.text(0.99, 0.01, '* frozen features', transform=ax.transAxes, ha='right', va='bottom', fontsize=7, color='grey')
    save(fig, name)


KEYM = lambda task: [m for m in [OURS, 'LinProbe phikon_v2 (frozen)', 'LogReg concat (frozen)', 'RBF-SVM concat (frozen)', 'MLP head concat (frozen)',
                                 'LinProbe dinov2_vitb14 (frozen)'] if m in OOF[task]]

for task in TASKS_DONE:
    sub = main[main.task == task]
    y = df[f'y_{task}'].values
    hbar(sub, 'bal_acc', f'F1_main_{task}', f'Balanced accuracy (%) — {task}, patient-disjoint 5-fold CV')
    hbar(sub, 'pat_bal_acc', f'F2_patient_{task}', f'Patient-level balanced accuracy (%) — {task}')
    # per-fold paired plot: ours vs best frozen
    best_c = sub[sub.method != OURS].groupby('method').bal_acc.mean().idxmax()
    a = sub[sub.method == OURS].groupby('fold').bal_acc.mean(); b = sub[sub.method == best_c].groupby('fold').bal_acc.mean()
    fig, ax = plt.subplots(figsize=(4.6, 3.8))
    for f in a.index:
        ax.plot([0, 1], [b[f] * 100, a[f] * 100], '-o', color='grey', alpha=0.6)
    ax.set_xticks([0, 1]); ax.set_xticklabels([short(best_c), 'Ours'], fontsize=8); ax.set_ylabel('Balanced accuracy (%)')
    ax.set_title(f'{task}: per-fold paired comparison', fontsize=9); save(fig, f'F13_per_fold_{task}')
    # reliability
    fig, ax = plt.subplots(figsize=(4.4, 4.4))
    for lab, P, col in [('raw', OOF_RAW[task][OURS], OKABE[1]), ('temperature-scaled', OOF[task][OURS], OKABE[0])]:
        conf, ok = P.max(1), P.argmax(1) == y
        xs, ys = zip(*[(conf[(conf > lo) & (conf <= hi)].mean(), ok[(conf > lo) & (conf <= hi)].mean())
                       for lo, hi in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]) if ((conf > lo) & (conf <= hi)).sum() >= 5] or [(np.nan, np.nan)])
        ax.plot(xs, ys, 'o-', color=col, label=f'Ours {lab} (ECE {ece_score(P, y):.3f})')
    ax.plot([0, 1], [0, 1], '--', color='grey'); ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.legend(frameon=False, fontsize=8)
    ax.set_title(f'Reliability — {task}', fontsize=9); save(fig, f'F12_reliability_{task}')
    # confusion (image + patient)
    P = OOF[task][OURS]; Pp, yp, _ = patient_agg(P, y, PATIENT); names = TASK_NAMES[task]
    fig, axes = plt.subplots(1, 2, figsize=(10.5 if task == 'subtype' else 7.5, 4.6 if task == 'subtype' else 3.4))
    for ax, (yt, pr, ttl) in zip(axes, [(y, P.argmax(1), 'image level'), (yp, Pp.argmax(1), 'patient level')]):
        cm = confusion_matrix(yt, pr, labels=range(len(names))); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45 if task == 'subtype' else 0); ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
        for i in range(len(names)):
            for j in range(len(names)):
                if cm[i, j]:
                    ax.text(j, i, f'{cm[i, j]}\n{cmn[i, j]*100:.0f}%', ha='center', va='center', fontsize=6.5, color='w' if cmn[i, j] > 0.5 else 'k')
        ax.set_title(f'Ours — {ttl}', fontsize=9); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    save(fig, f'F6_confusion_{task}')
    # ROC + PR
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
    for i, m in enumerate(KEYM(task)):
        Pm = OOF[task][m]
        if task == 'binary':
            fpr, tpr, _ = roc_curve(y, Pm[:, 1]); pr_, rc_, _ = precision_recall_curve(y, Pm[:, 1])
            auc_v, ap_v = roc_auc_score(y, Pm[:, 1]), average_precision_score(y, Pm[:, 1])
        else:                                   # micro-average one-vs-rest
            Y = np.eye(TASK_C[task])[y]
            fpr, tpr, _ = roc_curve(Y.ravel(), Pm.ravel()); pr_, rc_, _ = precision_recall_curve(Y.ravel(), Pm.ravel())
            auc_v, ap_v = roc_auc_score(Y.ravel(), Pm.ravel()), average_precision_score(Y.ravel(), Pm.ravel())
        lw = 2.2 if m == OURS else 1.1
        axes[0].plot(fpr, tpr, lw=lw, color=OKABE[i], label=f'{short(m)} (AUC {auc_v:.3f})')
        axes[1].plot(rc_, pr_, lw=lw, color=OKABE[i], label=f'{short(m)} (AP {ap_v:.3f})')
    axes[0].plot([0, 1], [0, 1], '--', color='grey'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    for a_ in axes:
        a_.legend(frameon=False, fontsize=7)
    axes[0].set_title(f'ROC — {task}' + (' (micro-avg)' if task == 'subtype' else ''), fontsize=9); axes[1].set_title('Precision–recall', fontsize=9)
    save(fig, f'F4_roc_pr_{task}')

# per magnification
if len(pm):
    for task in TASKS_DONE:
        t = pm[(pm.task == task) & pm.method.isin(KEYM(task))]
        fig, ax = plt.subplots(figsize=(6, 3.8))
        for i, (m, g) in enumerate(t.groupby('method', sort=False)):
            ax.plot(g.mag, g.bal_acc, '-o', lw=2.2 if m == OURS else 1.1, color=OKABE[i % 8], label=short(m))
        ax.set_xlabel('Magnification'); ax.set_ylabel('Balanced accuracy (%)'); ax.legend(frameon=False, fontsize=7)
        ax.set_title(f'Per-magnification performance — {task}', fontsize=9); save(fig, f'F7_per_magnification_{task}')
# per-class F1 (subtype)
if 'subtype' in TASKS_DONE:
    y = df.y_subtype.values
    fig, ax = plt.subplots(figsize=(7.5, 3.6)); w = 0.8 / len(KEYM('subtype'))
    for i, m in enumerate(KEYM('subtype')):
        f1s = f1_score(y, OOF['subtype'][m].argmax(1), labels=range(8), average=None, zero_division=0) * 100
        ax.bar(np.arange(8) + i * w, f1s, w, color=OKABE[i % 8], label=short(m))
    ax.set_xticks(np.arange(8) + 0.4 - w / 2); ax.set_xticklabels(SUBTYPES); ax.set_ylabel('F1 (%)'); ax.legend(frameon=False, fontsize=7, ncol=2)
    ax.set_title('Per-subtype F1 (pooled out-of-fold)', fontsize=9); save(fig, 'F15_per_class_f1_subtype')
# ablation
if 'ablation' in set(R.experiment):
    hbar(pd.concat([main[(main.task == 'subtype') & (main.method == OURS)], R[R.experiment == 'ablation']]), 'bal_acc', 'F8_ablation', 'Balanced accuracy (%) — subtype')
# external BACH
if 'external' in set(R.experiment) and EXT_META is not None:
    ext = R[R.experiment == 'external']; yx = EXT_META.y.values
    hbar(ext, 'bal_acc', 'F9_external_bach', 'Balanced accuracy (%) on BACH — trained on BreakHis only')
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0))
    for i, m in enumerate([m for m in [OURS, 'LinProbe phikon_v2 (frozen)', 'LogReg concat (frozen)', 'RBF-SVM concat (frozen)', 'MLP head concat (frozen)', ABL_NAME] if m in EXT_P]):
        Pm = EXT_P[m][min(EXT_P[m])]
        fpr, tpr, _ = roc_curve(yx, Pm[:, 1])
        axes[0].plot(fpr, tpr, lw=2.2 if m == OURS else 1.1, color=OKABE[i], label=f'{short(m)} (AUC {roc_auc_score(yx, Pm[:, 1]):.3f})')
    axes[0].plot([0, 1], [0, 1], '--', color='grey'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
    axes[0].legend(frameon=False, fontsize=7); axes[0].set_title('BACH external — ROC', fontsize=9)
    P0 = EXT_P[OURS][min(EXT_P[OURS])]
    cls4 = EXT_META.cls.str.lower().map({'normal': 'Normal', 'benign': 'Benign', 'insitu': 'InSitu', 'in situ': 'InSitu', 'invasive': 'Invasive'}).values
    rows4 = ['Normal', 'Benign', 'InSitu', 'Invasive']
    cm = np.array([[((cls4 == r) & (P0.argmax(1) == c)).sum() for c in (0, 1)] for r in rows4])
    cmn4 = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    axes[1].imshow(cmn4, cmap='Blues', vmin=0, vmax=1)
    for i in range(4):
        for j in range(2):
            axes[1].text(j, i, f'{cm[i, j]} ({cmn4[i, j]*100:.0f}%)', ha='center', va='center', fontsize=9, color='w' if cmn4[i, j] > 0.5 else 'k')
    axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['benign', 'malignant']); axes[1].set_xlabel('Predicted'); axes[1].set_yticks(range(4)); axes[1].set_yticklabels(rows4)
    axes[1].set_title('Ours on BACH: original 4 classes', fontsize=9); save(fig, 'F10_external_bach_roc_cm')
# leakage
if 'leakage' in set(R.experiment):
    lk = R[R.experiment == 'leakage'].groupby(['method', 'split']).bal_acc.mean().unstack() * 100
    fig, ax = plt.subplots(figsize=(5.8, 3.6)); x = np.arange(len(lk))
    for i, c_ in enumerate(lk.columns):
        bars = ax.bar(x + (i - 0.5) * 0.38, lk[c_], 0.38, label=c_, color=[OKABE[0], OKABE[3]][i])
        ax.bar_label(bars, fmt='%.1f', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([short(i) for i in lk.index]); ax.set_ylabel('Balanced accuracy (%)'); ax.set_ylim(0, 105)
    ax.legend(frameon=False, fontsize=8); ax.set_title('Effect of the split protocol (subtype)', fontsize=9); save(fig, 'F11_leakage')
# retrieval
for task in TASKS_DONE:
    rr = R[(R.experiment == 'retrieval') & (R.task == task)]
    if len(rr):
        hbar(rr, 'map_at_r', f'F16_retrieval_{task}', f'Cross-patient retrieval mAP@R (%) — {task}')
# saliency figures produced inside the training runs: copy them from attached outputs
for p in (INPUT.rglob('F14_saliency_*.p*') if INPUT.exists() else []):
    if not (FIG_DIR / p.name).exists():
        shutil.copy(p, FIG_DIR / p.name)

saved F1_main_binary
saved F2_patient_binary
saved F13_per_fold_binary
saved F12_reliability_binary
saved F6_confusion_binary
saved F4_roc_pr_binary
saved F7_per_magnification_binary
saved F9_external_bach
saved F10_external_bach_roc_cm
saved F16_retrieval_binary


## 18. Export

In [18]:
for d in ('results', 'tables', 'figures'):
    shutil.make_archive(str(WORK / f'paper_{d}_{RUN_PART}'), 'zip', root_dir=WORK, base_dir=d)
print('Done. Output tab: paper_tables_*.zip (CSV + LaTeX), paper_figures_*.zip (PNG + PDF), paper_results_*.zip (raw + bundle).')
print('Figures:', sorted(p.name for p in FIG_DIR.glob('*.png')))

Done. Output tab: paper_tables_*.zip (CSV + LaTeX), paper_figures_*.zip (PNG + PDF), paper_results_*.zip (raw + bundle).
Figures: ['F10_external_bach_roc_cm.png', 'F12_reliability_binary.png', 'F13_per_fold_binary.png', 'F14_saliency_binary.png', 'F16_retrieval_binary.png', 'F1_main_binary.png', 'F2_patient_binary.png', 'F4_roc_pr_binary.png', 'F6_confusion_binary.png', 'F7_per_magnification_binary.png', 'F9_external_bach.png']


## Paper map — which output supports which section
| Paper section | Output |
|---|---|
| Dataset & integrity | `T1_dataset`, `T2_integrity_audit` |
| Main results | `T3_main_binary`, `T4_main_subtype`, `F1_*`, `F2_*`, `F4_roc_pr_*`, `F6_confusion_*` |
| Statistical comparison | `T12_significance`, `F13_per_fold_*`, `claims_checklist.csv` |
| Ablation | `T5_ablation`, `F8_ablation` (+ ResNet-50 and LoRA-DINOv2 rows from the v3 run) |
| External validation | `T6_external_bach`, `F9_external_bach`, `F10_external_bach_roc_cm` |
| Robustness / analysis | `T9_per_magnification`, `F7_*`, `T10_per_class_*`, `F15_*`, `F12_reliability_*` |
| Leakage | `T7_leakage`, `F11_leakage` |
| Interpretability | `F14_saliency_*` |
| Embedding retrieval | `T8_retrieval`, `F16_*` |
| Compute | `T11_compute` |

**Rule:** every number in the paper must come from these files. Write a claim only if `claims_checklist.csv`
marks it SUPPORTED.